In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
names = open("names.txt").read().splitlines()

chars = sorted(list(set(''.join(names))))
stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0
itos = {i: ch for ch, i in stoi.items()}    
block_size = 5

def build_dataset(names):
    X, Y = [], []

    for w in names: 
        context = block_size * [0]
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(itos[i] for i in context), '->', itos[ix])
            context = context[1:] + [ix]
    

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(names)

n1 = int(.8 * len(names))
n2 = int(.9 * len(names))
Xtr, Ytr = build_dataset(names[:n1])
Xdev, Ydev = build_dataset(names[n1:n2])
Xtest, Ytest = build_dataset(names[n2:])


In [ ]:
# seed for replication
g = torch.Generator().manual_seed(2147483647)
# constants
vocab_size = len(stoi)
n_embed = 10

n_hidden = 200 

max_steps = 200000
batch_size = 32

# embeddings 
C = torch.randn(vocab_size, n_embed, generator=g)

# layer 1 
l1_size = block_size * n_embed
W1 = torch.randn(l1_size, n_hidden, generator=g) * (5/3)/(l1_size**.5)
#b1 = torch.randn(n_hidden, generator=g) * .01
epsilon = .00001

# layer 2
W2 = torch.randn(n_hidden, vocab_size, generator=g) * .01
b2 = torch.randn(vocab_size, generator=g) * 0

# batch norm
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

#parameters for gradient descent
parameters = [C, W1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True
lri = []
lossi = []
stepi = []


In [ ]:
for i in range(max_steps):
    #minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward pass 
    emb = C[Xb] #embed chars into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
    hpreact = embcat @ W1 #+b1 hidden layer pre-activation
    bnmean_i = hpreact.mean(0, keepdim=True)
    bnstd_i = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmean_i) / (bnstd_i + epsilon) + bnbias # standardize activations and shift/scale

    with torch.no_grad():
        bnmean_running = 0.999*bnmean_running + 0.001*bnmean_i
        bnstd_running = 0.999*bnstd_running + 0.001*bnstd_i
     
    h = torch.tanh(hpreact) # hidden layer activation
    logits = h @ W2 + b2   # out layer
    loss = F.cross_entropy(logits, Yb)
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if i < max_steps/2 else 0.01
    for p in parameters: 
        p.data -= lr * p.grad
    
    #track stats
    #lri.append(lre[i])
    lossi.append(loss.log10().item())
    stepi.append(i)

    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    #break
print(loss.item())


In [ ]:
plt.hist(h.view(-1).tolist(), 50)

In [ ]:
plt.hist(hpreact.view(-1).tolist(), 50)

In [ ]:
%matplotlib inline
plt.plot(stepi, lossi)

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 # + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnstd = hpreact.std(0, keepdim=True)

In [ ]:
bnmean

In [ ]:
bnmean_running

In [ ]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xtest, Ytest),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 #+ b1
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

### softmax, confidently wrong initialization fix
train 2.2015554904937744
val 2.270284652709961

### fix tanh from being too saturated
train 2.111660957336426
val 2.227128505706787

### batch norm + kaiming
train 1.9626660346984863
val 2.0393168926239014

In [ ]:
# visualize dimensions 0 and 1 of the embedding matrix C for all characters
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

In [ ]:
# sample from the model
#g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

In [ ]:
g = torch.Generator().manual_seed(2147483647)

class Linear: 
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight =  torch.randn(fan_in, fan_out, generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight 
        if self.bias is not None:
            self.out += self.bias 
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d: 

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.Training = True
        # params
        self.gamma = torch.ones((dim))
        self.beta = torch.zeros((dim))
        # buffers with running momentum
        self.running_mean = torch.zeros((dim))
        self.running_std = torch.ones((dim))


    def __call__(self, x):
        #calculate the forward pass
        if self.Training:
            x_mean = x.mean(0, keepdim=True)
            x_std = x.std(0, keepdim=True)
        else: 
            x_mean = self.running_mean
            x_std = self.running_std
        self.out = self.gamma * (x - x_mean) / (x_std + self.eps) + self.beta # standardize activations and shift/scale
        
        # update the buffers
        if self.Training:
            with torch.no_grad():
                self.running_mean = (1-self.momentum) * self.running_mean + self.momentum * x_mean
                self.running_stud = (1-self.momentum)* self.running_std + self.momentum * x_std
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh: 
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []
    
n_embd = 10 
n_hidden = 100

C = torch.randn(vocab_size, n_embd, generator=g) 

layers = [
    Linear(n_embd * block_size, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),       
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size), BatchNorm1d(vocab_size)
]

with torch.no_grad():
    # output softmax should be less confident
    layers[-1].gamma *= 0.1
    #layers[-1].weight *= 0.1
    # apply gain (expansion to counter squashing) to all other layers
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement()) for p in parameters)
for p in parameters: 
    p.requires_grad = True

In [ ]:
max_steps = 200000
batch_size=32
lossi=[]
ud = []

for i in range(max_steps):
    # batch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward pass
    emb = C[Xb] #embed chars into vectors
    x = emb.view(emb.shape[0], -1) # concatenate the vectors
    for layer in layers:
        x = layer(x) # pass the concatenated vectors through the layers
    loss = F.cross_entropy(x, Yb)

    # backwards pass
    for layer in layers:
        layer.out.retain_grad()
    for p in parameters:
        p.grad = None
    loss.backward()

    # update 
    lr = 0.1 if i < max_steps/2 else 0.01 # step learning rate decay 
    for p in parameters:
        p.data += -lr * p.grad
    
    # track stats
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())
    with torch.no_grad():
        ud.append([(lr*p.grad.std() / p.data.std()).log10().item() for p in parameters])
    if i > 1000:
        break

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('activation distribution')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('gradient distribution')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution');

In [ ]:
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:
    plt.plot([ud[j][i] for j in range(len(ud))])
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot
plt.legend(legends);